In [2]:
import pandas as pd
from itertools import combinations
from collections import Counter

# Load your datasets
articles_df = pd.read_csv("article_company_index.csv")
companies_df = pd.read_csv("nyse_nasdaq_companies_with_revenue_clenaed_JP.csv")

In [3]:
import ast

articles_df["company"] = articles_df["company"].apply(ast.literal_eval)
valid_companies = set(companies_df["company"].dropna())

In [4]:
target_companies = set(companies_df["company"].dropna())

In [5]:
pair_counter = Counter()

for _, row in articles_df.iterrows():
    mentioned_companies = set(row["company"])

    # Keep only companies that exist in your companies dataset
    matched_companies = mentioned_companies.intersection(target_companies)

    # Count each pair once per article
    for company_1, company_2 in combinations(sorted(matched_companies), 2):
        pair_counter[(company_1, company_2)] += 1

In [6]:
co_mentions_df = pd.DataFrame(
    [
        {
            "company_1": company_1,
            "company_2": company_2,
            "article_count": count
        }
        for (company_1, company_2), count in pair_counter.items()
    ]
)

co_mentions_df = co_mentions_df.sort_values(
    "article_count",
    ascending=False
).reset_index(drop=True)

co_mentions_df.head(20)

,company_1,company_2,article_count
0,Goldman Sachs,Morgan Stanley,58
1,Citigroup,Morgan Stanley,55
2,Amazon,Walmart,50
3,Broadcom,Qualcomm,48
4,Amazon,Microsoft,45
5,Deutsche Bank,Morgan Stanley,43
6,Deutsche Bank,UBS,40
7,Deutsche Bank,Goldman Sachs,40
8,Citigroup,Goldman Sachs,31
9,HSBC,UBS,29


In [7]:
co_mentions_df.to_csv("company_co_mentions.csv", index=False)

In [8]:
import pandas as pd
import networkx as nx
from ipysigma import Sigma
from networkx.algorithms.community import louvain_communities

# Load files
co_mentions_df = pd.read_csv("company_co_mentions.csv")
companies_df = pd.read_csv("nyse_nasdaq_companies_with_revenue_clenaed_JP.csv")

# Create graph
G = nx.Graph()

for _, row in co_mentions_df.iterrows():
    G.add_edge(
        row["company_1"],
        row["company_2"],
        weight=row["article_count"]
    )

# Add basic node attributes
for node in G.nodes():
    G.nodes[node]["label"] = node

weighted_degree = dict(G.degree(weight="weight"))
nx.set_node_attributes(G, weighted_degree, "weighted_degree")

In [9]:
industry_group_terms = {
    "Finance": [
        "financial services",
        "financial service activities, except insurance and pension funding",
        "finance",
        "economics of banking",
        "bank",
        "investment",
        "asset management",
        "insurance",
        "insurance industry",
        "life insurance",
        "vehicle insurance",
        "health insurance",
        "health insurance company",
        "risk management",
        "wire transfer",
        "Bitcoin",
        "International Standard Industrial Classification",
    ],

    "Energy": [
        "petroleum industry",
        "energy industry",
        "energy company",
        "industrial gas",
    ],

    "Utilities": [
        "public utility",
        "electricity supply company",
        "electricity generation",
        "electric power industry",
        "water supply",
    ],

    "Technology / Telecom": [
        "software industry",
        "software development",
        "enterprise software",
        "information technology",
        "information technology industry",
        "information technology consulting",
        "information and communications technology",
        "computer security",
        "information security",
        "computer and network surveillance",
        "computer industry",
        "computer hardware industry",
        "computer network",
        "computer storage media",
        "computer-aided design",
        "networking hardware",
        "Internet",
        "web hosting service",
        "technology",
        "technology company",
        "artificial intelligence",
        "analytics",
        "automation",
        "robotics",
        "3D printing",
        "semiconductor industry",
        "electronics",
        "consumer electronics industry",
        "telecommunications",
        "communication",
        "video conference",
        "telepresence",
        "digital distribution",
        "electrical industry",
    ],

    "Healthcare": [
        "pharmaceutical industry",
        "biotechnology",
        "biotechnology industry",
        "health care",
        "health technology",
        "medical technology industry",
        "medical equipment",
        "managed care",
        "life sciences",
        "high-performance liquid chromatography",
    ],

    "Automotive": [
        "automotive industry",
        "car rental company",
    ],

    "Industrials / Aerospace / Defense": [
        "industrial manufacturing",
        "industrial sector",
        "mechanical engineering",
        "engineering",
        "aerospace industry",
        "aerospace engineering",
        "aviation",
        "weapons industry",
        "defense contractor",
        "manufacture of machinery and equipment",
        "equipment rental",
        "power tool",
        "shipbuilding",
        "construction",
        "facility management",
        "outsourcing",
    ],

    "Materials / Mining / Chemicals": [
        "chemical industry",
        "Pesticide and Other Agricultural Chemical Manufacturing",
        "iron and steel industry",
        "mining",
        "mining industry",
        "metal",
        "cement industry",
        "building materials trade",
        "glass",
        "pulp and paper industry",
    ],

    "Consumer / Retail / Food": [
        "retail",
        "wholesale",
        "trade",
        "e-commerce",
        "direct selling",
        "auction",
        "product distribution",
        "final good",
        "hardware store",
        "fast-moving consumer goods",
        "personal care product",
        "cosmetics industry",
        "food industry",
        "food processing",
        "food service",
        "restaurant",
        "fast food",
        "fast casual restaurant",
        "system catering",
        "beverage industry",
        "brewing industry",
        "coffee industry",
        "manufacture of cocoa, chocolate and sugar confectionery",
        "alcohol industry",
        "tobacco industry",
        "clothing industry",
        "footwear industry",
        "textile industry",
        "cannabis industry",
    ],

    "Media / Entertainment": [
        "media industry",
        "mass media",
        "show business",
        "streaming media",
        "broadcasting",
        "broadcast television system",
        "television",
        "terrestrial television",
        "radio broadcasting",
        "journalism",
        "music industry",
        "production music",
        "animation",
        "video game industry",
        "game industry",
        "sports industry",
        "gambling",
        "gambling industry",
    ],

    "Transport / Logistics": [
        "logistics",
        "transport",
        "transport industry",
        "freight transport industry",
        "air transport",
        "rail transport",
        "water transport",
        "shipping line",
        "waste management",
        "waste management industry",
    ],

    "Real Estate": [
        "real estate industry",
        "real estate investment trust",
        "self storage",
    ],

    "Travel / Hospitality": [
        "tourism",
        "tourism industry",
        "hospitality industry",
        "space tourism",
    ],

    "Agriculture": [
        "agriculture",
        "agribusiness",
    ],

    "Professional Services": [
        "professional service",
        "consulting company",
        "marketing",
        "E-recruitment",
    ],

    "Holding / Conglomerate": [
        "holding company",
        "holding company activities",
        "conglomerate",
    ],

    "Other / Unclear": [
        "tertiary sector of the economy",
        "quaternary sector of the economy",
    ],
}
def map_to_industry_group(industry):
    if pd.isna(industry):
        return "Unknown"

    industry = str(industry).strip()

    for group, terms in industry_group_terms.items():
        if industry in terms:
            return group

    return "Other / Unmapped"
companies_df["industry_group"] = companies_df["industry"].apply(map_to_industry_group)
# Add industry to graph nodes

industry_map = dict(
    zip(companies_df["company"], companies_df["industry"])
)
industry_group_map = dict(
    zip(companies_df["company"], companies_df["industry_group"])
)

for node in G.nodes():
    G.nodes[node]["industry_group"] = industry_group_map.get(node, "Unknown")

In [10]:
communities = louvain_communities(
    G,
    weight="weight",
    resolution=1,
    seed=42
)

for cluster_id, community in enumerate(communities):
    for company in community:
        G.nodes[company]["cluster"] = cluster_id

print(f"Found {len(communities)} clusters")

Found 13 clusters


In [12]:
Sigma(
    G,
    node_color="cluster",
    node_label="label",
    node_size="weighted_degree",
    node_size_range=(4, 30),
    edge_size="weight",
    edge_size_range=(1, 8),
    height=800,
    start_layout=True
)

Sigma(nx.Graph with 234 nodes and 1,003 edges)

In [14]:
Sigma(
    G,
    node_color="industry_group",
    node_label="label",
    node_size="weighted_degree",
    node_size_range=(4, 30),
    edge_size="weight",
    edge_size_range=(1, 8),
    height=800,
    start_layout=True
)

Sigma(nx.Graph with 234 nodes and 1,003 edges)

In [15]:
node_rows = []

for node, attrs in G.nodes(data=True):
    node_rows.append({
        "company": node,
        "cluster": attrs.get("cluster"),
        "industry": attrs.get("industry"),
        "industry_group": attrs.get("industry_group"),
        "weighted_degree": attrs.get("weighted_degree"),
        "degree": G.degree(node)
    })

nodes_df = pd.DataFrame(node_rows)

nodes_df.head()

,company,cluster,industry,industry_group,weighted_degree,degree
0,Goldman Sachs,0,None,Finance,319,59
1,Morgan Stanley,0,None,Finance,382,62
2,Citigroup,0,None,Finance,254,52
3,Amazon,1,None,Consumer / Retail / Food,301,63
4,Walmart,1,None,Consumer / Retail / Food,107,27


In [16]:
cluster_industry_table = pd.crosstab(
    nodes_df["cluster"],
    nodes_df["industry_group"]
)

cluster_industry_percent = cluster_industry_table.div(
    cluster_industry_table.sum(axis=1),
    axis=0
)

cluster_industry_percent.round(2)

industry_group,Agriculture,Automotive,Consumer / Retail / Food,Energy,Finance,Healthcare,Holding / Conglomerate,Industrials / Aerospace / Defense,Materials / Mining / Chemicals,Media / Entertainment,Real Estate,Technology / Telecom,Transport / Logistics,Travel / Hospitality,Unknown,Utilities
cluster,,,,,,,,,,,,,,,,
0,0.00,0.00,0.00,0.06,0.60,0.00,0.00,0.06,0.03,0.00,0.03,0.06,0.00,0.00,0.11,0.06
1,0.00,0.00,0.17,0.00,0.14,0.00,0.00,0.06,0.00,0.09,0.02,0.44,0.02,0.03,0.02,0.03
2,0.00,0.00,0.11,0.00,0.00,0.11,0.00,0.11,0.00,0.00,0.00,0.67,0.00,0.00,0.00,0.00
3,0.00,0.00,0.07,0.04,0.07,0.11,0.04,0.32,0.07,0.00,0.00,0.07,0.11,0.04,0.07,0.00
4,0.00,0.47,0.12,0.00,0.00,0.00,0.00,0.06,0.00,0.06,0.00,0.29,0.00,0.00,0.00,0.00
5,0.02,0.00,0.25,0.21,0.04,0.29,0.00,0.00,0.08,0.00,0.00,0.10,0.00,0.00,0.02,0.00
6,0.00,0.00,0.07,0.07,0.07,0.00,0.00,0.00,0.14,0.00,0.00,0.50,0.00,0.00,0.14,0.00
7,0.00,0.00,0.00,0.00,0.50,0.00,0.00,0.00,0.00,0.00,0.00,0.50,0.00,0.00,0.00,0.00
8,0.00,0.00,0.00,0.00,0.00,0.50,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.50


In [17]:
cluster_summary = []

for cluster_id, group in nodes_df.groupby("cluster"):
    industry_counts = group["industry_group"].value_counts()

    top_industry = industry_counts.index[0]
    top_count = industry_counts.iloc[0]
    cluster_size = len(group)

    cluster_summary.append({
        "cluster": cluster_id,
        "top_industry": top_industry,
        "top_industry_count": top_count,
        "cluster_size": cluster_size,
        "purity": top_count / cluster_size
    })

cluster_summary_df = pd.DataFrame(cluster_summary)

cluster_summary_df = cluster_summary_df.sort_values(
    "purity",
    ascending=False
)

cluster_summary_df

,cluster,top_industry,top_industry_count,cluster_size,purity
10,10,Industrials / Aerospace / Defense,2,2,1.000000
9,9,Technology / Telecom,2,2,1.000000
12,12,Consumer / Retail / Food,2,3,0.666667
2,2,Technology / Telecom,6,9,0.666667
0,0,Finance,21,35,0.600000
7,7,Finance,1,2,0.500000
8,8,Utilities,1,2,0.500000
11,11,Unknown,1,2,0.500000
6,6,Technology / Telecom,7,14,0.500000
4,4,Automotive,8,17,0.470588


In [18]:
ct = pd.crosstab(nodes_df["cluster"], nodes_df["industry_group"])

overall_purity = ct.max(axis=1).sum() / ct.sum().sum()

print("Overall cluster purity:", round(overall_purity, 3))

Overall cluster purity: 0.444
